In [24]:
from langgraph_sdk import get_client
import httpx

def get_langgraph_client():
    URL = "http://127.0.0.1:2024"
    client = get_client(url=URL)
    if client is None:
        raise Exception("Failed to connect to LangGraph server")
    try:
        client.http.client.timeout = httpx.Timeout(connect=5, read=30*60, write=5*60, pool=5)
    except Exception as e:
        print(f"Failed to set timeout for LangGraph client: {e}")
        return client
    return client

async def get_assistant(client):
    try:
        assistant = await client.assistants.search(limit=25)
        return assistant
    except Exception as e:
        raise Exception(f"Failed to get assistant: {e}")
    

_thread_id = None    
async def get_or_create_thread(client):
    global _thread_id
    _thread_id = None  
    if _thread_id is None:
        try:
            _thread_id = await client.threads.create()
            _thread_id = _thread_id['thread_id']
            print(f"Created thread: {_thread_id}")
        except Exception as e:
            print(f"Failed to create thread: {e}")
            return None
    return _thread_id


In [25]:
from langchain_core.messages import HumanMessage

async def run_jd_generator(client, input_string, _thread_id):
    input_data = {
        "messages": [
            HumanMessage(content=input_string)
        ]
    }
    selected_agent = "33f5bbdc-e7a1-562f-b4e5-b620117e1da5"
    final_state = await client.runs.wait(
        config={"recursion_limit": 200},
        thread_id=_thread_id,
        assistant_id=selected_agent,
        input=input_data
    )
    return final_state


In [26]:
client = get_langgraph_client()

In [27]:
assistants = await get_assistant(client)
for assistant in assistants:
    print(f"Assistant name: {assistant['graph_id']}, ID: {assistant['assistant_id']}")

Assistant name: business_context_analyzer, ID: 7dec50d8-eb0a-5769-94e1-83e86449f127
Assistant name: job_description_generator, ID: 33f5bbdc-e7a1-562f-b4e5-b620117e1da5


In [32]:
_thread_id = await get_or_create_thread(client)
input_string = f"""
  {{
  "company_name": "Innovatech Solutions",
  "company_description": "Innovatech Solutions is a rapidly growing SaaS company specializing in AI-driven platforms that help businesses optimize their workflows and enhance customer engagement.",
  "job_title": "Full-Stack Software Engineer",
  "required_experience": "3+ years of professional software development experience in both frontend and backend technologies.",
  "required_technical_skills": [
    "Proficiency in JavaScript and React.js",
    "Strong backend experience with Node.js and Express",
    "Familiarity with databases such as PostgreSQL and MongoDB",
    "Experience with RESTful APIs and microservices architecture",
    "Version control with Git and CI/CD pipelines (e.g., GitHub Actions, Jenkins)",
    "Cloud deployment experience (AWS, Azure, or GCP)"
  ],
  "soft_skills": [
    "Strong problem-solving and analytical thinking",
    "Excellent communication and teamwork abilities",
    "Adaptability in a fast-paced agile environment",
    "Attention to detail and a proactive mindset"
  ],
  "working_conditions": "Hybrid work model (3 days on-site, 2 days remote); flexible working hours; modern office in the Melbourne CBD with ergonomic workstations and a collaborative environment.",
  "education_requirements": "Bachelor's degree in Computer Science, Software Engineering, or a related field. A Master's degree or relevant certifications (e.g., AWS Certified Developer) is a plus.",
  "key_responsibilities": [
    "Design, develop, and maintain scalable web applications using React and Node.js",
    "Collaborate with cross-functional teams including product managers, designers, and QA engineers",
    "Write clean, maintainable, and efficient code following best practices",
    "Participate in code reviews and mentor junior developers",
    "Identify and troubleshoot software issues and performance bottlenecks",
    "Contribute to architectural decisions and the overall technical direction of the product"
  ]
}}
"""

jd_output = await run_jd_generator(client, input_string, _thread_id)

Created thread: 72596ca8-6143-46e9-89dd-60af35e74f21


In [45]:
print(jd_output["formatted_job_description"]['content'])

**Job Title:** Full-Stack Software Engineer

---

**About the Company**  
Innovatech Solutions is a forward-thinking technology company specializing in AI-driven SaaS platforms. We are dedicated to delivering innovative, scalable software solutions that empower businesses to harness the power of artificial intelligence. Our collaborative and dynamic environment fosters growth, creativity, and continuous learning.

---

**Job Summary**  
We are seeking a talented Full-Stack Software Engineer to join our development team. In this role, you will design, develop, and maintain cutting-edge web applications, contributing to the evolution of our AI-powered SaaS platform. The ideal candidate will bring at least 3 years of full-stack experience, a passion for technology, and a collaborative spirit.

---

**Key Responsibilities**  
- Design, develop, and maintain scalable web applications using React.js and Node.js.  
- Collaborate with cross-functional teams including product managers, designer